expanding/exploring 2.1 stuff : )
also testing out mlflow for tracking because its getting hectic

In [1]:
import pandas as pd
import numpy as np
from factor_analyzer import FactorAnalyzer
from src.features import feature_encoder, raw_data_loader
import os 

In [2]:
os.environ["MLFLOW_EXPERIMENT_NAME"] = "2.1-jp-feature-engineering"

In [3]:
df = raw_data_loader.load_and_clean_raw("../")

df = feature_encoder.encode_features(df)

In [4]:
df.head()

,age,bmi,sex,sw_9am_start_diff,sw_5pm_end_diff,nasal_congestion_stuffiness_nose,nasal_blockage_obstr_nose,troub_brth_nose,troub_slp_nose,not_enough_air_excercise_nose,...,pulmonary_problem_other_mdhx,chronic_obstructive_pulmonary_disease_mdhx,asthma_mdhx,cardiovascular_problem_other_mdhx,congestive_heart_failure_mdhx,hypertension_mdhx,oophorectomy_bilateral_mdhx,ahi,dream_recall_frequency_infrequent,dream_recall_frequency_rarely_or_never
0,58.0,30.7,1.0,-1.00,0.0,1.0,0.0,0.0,2.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.3,0.0,1.0
1,30.0,29.4,1.0,-1.00,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.4,1.0,0.0
2,30.0,25.8,1.0,-2.00,2.0,0.0,0.0,0.0,4.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
3,42.0,26.8,0.0,-3.00,3.0,1.0,0.0,0.0,3.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.9,0.0,1.0
4,36.0,45.2,0.0,-0.75,0.5,3.0,1.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.3,0.0,1.0


In [7]:
import mlflow

mlflow.set_experiment(os.getenv("MLFLOW_EXPERIMENT_NAME", "default_experiment"))
mlflow.set_tracking_uri("file:///Users/jack/Repos/apnea-predictor/mlruns")


2026/04/17 12:55:10 INFO mlflow.tracking.fluent: Experiment with name '2.1-jp-feature-engineering' does not exist. Creating a new experiment.


In [8]:
from src.features.transformers import Factor_Analyzer_Transformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from src.utils.data_utils import convert_ahi
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, f1_score


fa_transformer = Factor_Analyzer_Transformer(n_factors=18, rotation="promax")


X = df.drop(columns=["ahi"])
y = convert_ahi(df["ahi"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

train_data = pd.concat([X_train, y_train], axis=1)

train_dataset = mlflow.data.from_pandas(
    train_data, name="2.1-sleep-apnea-train-raw", targets="ahi"
)

test_dataset = mlflow.data.from_pandas(
    pd.concat([X_test, y_test], axis=1), name="2.1-sleep-apnea-test-raw", targets="ahi"
)


with mlflow.start_run():
    mlflow.log_input(train_dataset, context="train")
    mlflow.log_input(test_dataset, context="test")
    params = {"objective": "reg:squarederror", "max_depth": 6}
    mlflow.log_params(params)
    model = XGBClassifier(**params)
    model = model.fit(X_train, y_train)
    
    mlflow.xgboost.log_model(
        xgb_model=model,
        name="xgb_model",
        model_format="json",
    )

    y_pred = model.predict(X_test)
    mlflow.log_metric("test_accuracy", np.mean(y_pred == y_test))
    mlflow.log_metric("test_auc", roc_auc_score(y_test, y_pred))
    mlflow.log_metric("test_f1", f1_score(y_test, y_pred))
    

/Users/jack/Repos/apnea-predictor/conda_env/lib/python3.14/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
